In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
import os
import pandas as pd
from boxsdk import Client, CCGAuth
from dotenv import load_dotenv
from pathlib import Path
import os
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.query import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *

load_dotenv(override=True)

CLIENT_ID = os.getenv("BOXCLIENTID")
CLIENT_SECRET = os.getenv("BOXCLIENTSECRET")
assert CLIENT_ID and CLIENT_SECRET, "Set BOXCLIENTID and BOXCLIENTSECRET in your environment"

auth = CCGAuth(client_id=CLIENT_ID, client_secret=CLIENT_SECRET, enterprise_id=20888)
client = Client(auth)

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
FOLDER_ID = "370739222165"  # EU-ANALYTICAL-CUSTOMER-OUTPUT


def list_folder_items(client, folder_id, limit=1000):
    folder = client.folder(folder_id).get()
    items = []
    offset = 0
    while True:
        batch = list(
            client.folder(folder_id).get_items(
                limit=limit,
                offset=offset,
                fields=["id", "name", "type", "size", "modified_at", "created_at"],
            )
        )
        if not batch:
            break
        items.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
    return folder, items


folder, items = list_folder_items(client, FOLDER_ID)
print(f"Folder: {folder.name} (id={folder.id})")
print(f"Total items: {len(items)}")

Folder: EU-ANALYTICAL-CUSTOMER-OUTPUT (id=370739222165)
Total items: 38


In [4]:
EXCLUDED_FOLDERS = {
    "ANALYZERS-LOCATION-TRACKING",
    "CUSTOMER GLOBAL LEAK EXTRACTIONS",
    "GAS-DATA-BOT-EU-OUTPUT",
    "MASTER VIEW REPORTS"
}

rows = [
    {
        "type": item.type,
        "id": item.id,
        "folder_name": item.name,
        "size": getattr(item, "size", None),
        "modified_at": getattr(item, "modified_at", None),
        "created_at": getattr(item, "created_at", None),
    }
    for item in sorted(items, key=lambda x: (x.type, x.name.lower()))
    if item.name not in EXCLUDED_FOLDERS
]

folder_df = pd.DataFrame(rows)
#folder_df

In [ ]:
KPI_SUBFOLDER_NAME = "KPI"
SKIP_KPI_FOLDERS = {"UK-CADENT"}


def get_or_create_subfolder(client, parent_folder_id, subfolder_name):
    parent = client.folder(parent_folder_id)
    for item in parent.get_items(limit=1000, fields=["id", "name", "type"]):
        if item.type == "folder" and item.name == subfolder_name:
            return item.id
    return parent.create_subfolder(subfolder_name).id


kpi_folder_ids = []
for _, row in folder_df.iterrows():
    if row["folder_name"] in SKIP_KPI_FOLDERS:
        kpi_folder_ids.append('375691804502')
        #print(f"Skipped: {row['name']}")
        continue

    kpi_id = get_or_create_subfolder(client, row["id"], KPI_SUBFOLDER_NAME)
    kpi_folder_ids.append(kpi_id)
   #print(f"{row['name']}: KPI folder id = {kpi_id}")

folder_df["kpi_folder_id"] = kpi_folder_ids

# Split the "name" column at the first "-" into "country" and "customer" columns
folder_df[["country", "customer"]] = folder_df["folder_name"].str.split("-", n=1, expand=True)

CUSTOMER_ALIASES = {
    "WALES & WEST UTILITIES": "WALES AND WEST UTILITIES",
}


def normalize_customer(name):
    if pd.isna(name):
        return name
    key = str(name).strip().upper()
    key = CUSTOMER_ALIASES.get(key, key)
    return key.upper()


kpi_customer = Query(
    "SELECT CustomerId, Name FROM KPI_Customer"
).execute(KPIHub_Conn)

folder_df["customer_key"] = folder_df["customer"].map(normalize_customer)
kpi_customer["customer_key"] = kpi_customer["Name"].map(normalize_customer)

folder_df = pd.merge(
    folder_df,
    kpi_customer.rename(columns={"Name": "NameKPI"}),
    on="customer_key",
    how="left",
).drop(columns=["customer_key"])


In [16]:
folder_df

,type,id,folder_name,size,modified_at,created_at,kpi_folder_id,country,customer,CustomerId,NameKPI
0,folder,316829663240,AUSTRIA-NETZ NIEDERÖSTERREICH,830679734,2026-08-25T07:01:01-07:00,2025-04-15T08:02:05-07:00,398256904676,AUSTRIA,NETZ NIEDERÖSTERREICH,7460AB4D-648E-297D-6073-3A145CB49DBC,Netz Niederösterreich
1,folder,289226673282,AZERBAIJAN-AZERIGAS,3683249581,2026-08-25T06:57:13-07:00,2024-10-15T07:12:16-07:00,411868607004,AZERBAIJAN,AZERIGAS,DFA4A93D-3DA3-8183-D313-3A14AAFE01AD,Azerigas
2,folder,276270484038,CZECH REPUBLIC-PPD,1494264,2026-08-25T06:57:14-07:00,2024-07-22T20:48:35-07:00,411852236642,CZECH REPUBLIC,PPD,D9ECCD39-EE06-899B-80FA-3A13237051F4,PPD
3,folder,281481453536,GERMANY-AVACON,9608505570,2026-08-25T07:00:45-07:00,2024-08-22T08:06:56-07:00,398246560432,GERMANY,AVACON,ED346655-AAF7-B2E7-43E2-3A10E455E700,Avacon
4,folder,281481465536,GERMANY-E-NETZ SUDHESSEN,178526814,2026-08-21T06:39:24-07:00,2024-08-22T08:06:57-07:00,398249781410,GERMANY,E-NETZ SUDHESSEN,E5F6C480-6796-9A6E-79DE-3A0EF1744B1C,E-NETZ SUDHESSEN
5,folder,281528732442,GERMANY-ENBW,309074590,2026-08-25T03:45:23-07:00,2024-08-22T15:11:25-07:00,398247368526,GERMANY,ENBW,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW
6,folder,365163782325,GERMANY-ENERGIEVERSORGUNG FILSTAL,10611279,2026-08-25T08:10:51-07:00,2026-02-09T05:27:49-08:00,398250462372,GERMANY,ENERGIEVERSORGUNG FILSTAL,5DDE747E-0AAD-B45F-8039-3A1897DBC43F,Energieversorgung Filstal
7,folder,338415047951,GERMANY-EWE,483963120,2026-08-25T07:01:35-07:00,2025-08-29T03:53:30-07:00,398251266011,GERMANY,EWE,7D9E66E1-6EE8-27D6-76B4-3A10E451DD30,EWE
8,folder,276411078439,GERMANY-N-ERGIE,2408842,2026-08-25T06:57:18-07:00,2024-07-23T16:14:52-07:00,411852685386,GERMANY,N-ERGIE,696A3AB7-C666-6E14-FF0C-3A11B2E3C4D4,N-ERGIE
9,folder,281481484736,GERMANY-NBB,385716870,2026-08-21T06:39:08-07:00,2024-08-22T08:06:58-07:00,398247366692,GERMANY,NBB,A1E8BEC0-6A89-9454-137B-3A0A41825A09,NBB


In [15]:
output = folder_df[["CustomerId", "kpi_folder_id"]].dropna()
output['LastUpdated'] = datetime.now()
output.rename(columns = {'kpi_folder_id': 'BoxFolderId'}, inplace = True)
KPI_OutputExcelLocation.update_table(arguments = {'DataFrame': output, 'db_path': DB_PATH, 'PrimaryKey': 'CustomerId'})